In [0]:
from datetime import *
from delta.tables import *
from functools import reduce

from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window as W
from pyspark.storagelevel import StorageLevel

import os
import time
import re
import json

# Get current UTC time to be used for entire batch of files to be ingested
dtUTC = datetime.utcnow()
BATCH_TIME = dtUTC.isoformat() # UTC timestamp for type 2 column
BATCH_TIME_STR = dtUTC.strftime('%Y-%m-%dT%H:%M:%S.%fZ') # UTC timestamp for MERGE commands

# Output variables
success = True
data_changed = 0
output_message = ""
table_exists = False


In [0]:
class DeltaProcessor:

    def __init__(self, schema_name=None, table_name=None):
        self.schema_name = schema_name
        self.table_name = table_name
    
    # Checking DB exists or not. If not, create new database.
    def schema_exists(self, create_if_not_exists=False, delta_location=None):
        self.schema_name = self.schema_name.lower()
        try:
            spark.sql(f"DESCRIBE DATABASE {self.schema_name}")
            print("Database/schema is already exists")
            return True
        except Exception:
            create_if_not_exist = True
            if create_if_not_exist:
                if delta_location is not None:
                    delta_location = f" MANAGED LOCATION '{delta_location}'"
                try:
                    print("Created database/schema as it is not exists")
                    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {self.schema_name} {delta_location}")
                    return True
                except Exception as e:
                    print(f"Error creating schema: {e}")
                    return False
            return False
    
    # Check table exist within the database or not.
    def table_exists(self):
        try:
            spark.sql(f"DESCRIBE TABLE {self.schema_name}.{self.table_name}")
            print(f"{self.schema_name}.{self.table_name} is exists")
            return True
        except Exception:
            print(f"{self.schema_name}.{self.table_name} is not exists")
            return False

    # Drop table from database.    
    def drop_table(self, drop_external_data=False):
        if self.table_exists():
            table_properties = spark.sql(f"DESCRIBE FORMATTED {self.schema_name}.{self.table_name}")
            
            try:
                location = table_properties.filter(table_properties.col_name == "Location").collect()[0]["data_type"]
            except Exception as e:
                print(str(e))

            print(f"Dropping Table: {self.schema_name}.{self.table_name}")
            spark.sql(f"DROP TABLE {self.schema_name}.{self.table_name}")

            if drop_external_data and location:
                print(f"Dropping Location: {location}")
                dbutils.fs.rm(location, recurse=True)
        else:
            print(f"Table {self.schema_name}.{self.table_name} does not exist.")